In [2]:
from bs4 import BeautifulSoup as bs
import requests
import json
import re
from tqdm import tqdm

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import urllib

import time
import unicodedata

### First part trying

Genius n'est pas bon car l'entiéreté de la discographie n'est pas chargée par défaut durant la génération de la page initiale, ce qui empêche de pouvoir collecter de manière automatique l'entiéreté de la discographie d'un artiste. Choix refusé

In [29]:
request = requests.get(url="https://www.deezer.com/fr/artist/14659541/top_track")
html_page = request.text


In [30]:
page = bs(html_page,"html.parser")


In [ ]:
#L'entiéreté de la discographie de l'artiste est contenue dans une gigantesque partie script 
res = page.find("script", string = re.compile("ART_ID"))

In [9]:
#La partie qui nous intéresse à l'intérieur de ce script est 
xx = res.find(string=re.compile("TOP"))
print(xx)

window.__DZR_APP_STATE__ = {"DATA":{"ART_ID":"14659541","ART_NAME":"Limsa d'Aulnay","URL_REWRITING":"limsa-d-aulnay","ART_PICTURE":"fdf46f8f129ba2d3897e94cfa9bc80f9","NB_FAN":49939,"URL":"","TWITTER":"","STATUS":{"status":"","date":"2024-11-13 11:48:56"},"USER_ID":"0","DATA":{"locales":{"lang_fr":{"name":"Limsa d'Aulnay"}},"COPYRIGHT_PICTURE":""},"__TYPE__":"artist","SMARTRADIO":true},"TOP":{"data":[{"SNG_ID":"2171552607","PRODUCT_TRACK_ID":"1451835217","UPLOAD_ID":0,"SNG_TITLE":"Trucs sentimentaux","ART_ID":"67211132","PROVIDER_ID":"19","ART_NAME":"BEN plg","ARTIST_IS_DUMMY":false,"ARTISTS":[{"ART_ID":"67211132","ROLE_ID":"0","ARTISTS_SONGS_ORDER":"11","ART_NAME":"BEN plg","ARTIST_IS_DUMMY":false,"ART_PICTURE":"9bfad6b2143d4081def8b19766531890","RANK":"724123","LOCALES":{"lang_fr":{"name":"BEN plg"}},"__TYPE__":"artist"},{"ART_ID":"14659541","ROLE_ID":"0","ARTISTS_SONGS_ORDER":"12","ART_NAME":"Limsa d'Aulnay","ARTIST_IS_DUMMY":false,"ART_PICTURE":"fdf46f8f129ba2d3897e94cfa9bc80f9","RA

In [10]:
vvv = res.text.split("window.__DZR_APP_STATE__ = ")[1]
structured_text = json.loads(vvv)
structured_text

{'DATA': {'ART_ID': '14659541',
  'ART_NAME': "Limsa d'Aulnay",
  'URL_REWRITING': 'limsa-d-aulnay',
  'ART_PICTURE': 'fdf46f8f129ba2d3897e94cfa9bc80f9',
  'NB_FAN': 49939,
  'URL': '',
  'TWITTER': '',
  'STATUS': {'status': '', 'date': '2024-11-13 11:48:56'},
  'USER_ID': '0',
  'DATA': {'locales': {'lang_fr': {'name': "Limsa d'Aulnay"}},
   'COPYRIGHT_PICTURE': ''},
  '__TYPE__': 'artist',
  'SMARTRADIO': True},
 'TOP': {'data': [{'SNG_ID': '2171552607',
    'PRODUCT_TRACK_ID': '1451835217',
    'UPLOAD_ID': 0,
    'SNG_TITLE': 'Trucs sentimentaux',
    'ART_ID': '67211132',
    'PROVIDER_ID': '19',
    'ART_NAME': 'BEN plg',
    'ARTIST_IS_DUMMY': False,
    'ARTISTS': [{'ART_ID': '67211132',
      'ROLE_ID': '0',
      'ARTISTS_SONGS_ORDER': '11',
      'ART_NAME': 'BEN plg',
      'ARTIST_IS_DUMMY': False,
      'ART_PICTURE': '9bfad6b2143d4081def8b19766531890',
      'RANK': '724123',
      'LOCALES': {'lang_fr': {'name': 'BEN plg'}},
      '__TYPE__': 'artist'},
     {'ART_ID':

In [11]:
for i in structured_text :
    print(i)

DATA
TOP
HIGHLIGHT
BIO
SELECTED_PLAYLIST
RELATED_PLAYLIST
RELATED_ARTISTS
VIDEO
ALBUMS


In [12]:
for i in structured_text["TOP"]:
    print(i)

data
count
total
version
filtered_count
filtered_items
next


In [13]:
print(f'Number of track by author : {structured_text["TOP"]["total"]}')

Number of track by author : 62


In [14]:
part_to_parse = structured_text["TOP"]["data"]
len(part_to_parse)

62

In [47]:
title_list[-14]

['2 spliffs', True]

In [48]:
part_to_parse[-14]

{'SNG_ID': '1714751297',
 'PRODUCT_TRACK_ID': '1196771957',
 'UPLOAD_ID': 0,
 'SNG_TITLE': '2 spliffs',
 'ART_ID': '209317037',
 'PROVIDER_ID': '452',
 'ART_NAME': "L'agence records",
 'ARTIST_IS_DUMMY': False,
 'ARTISTS': [{'ART_ID': '209317037',
   'ROLE_ID': '0',
   'ARTISTS_SONGS_ORDER': '1',
   'ART_NAME': "L'agence records",
   'ARTIST_IS_DUMMY': False,
   'ART_PICTURE': 'e4a3578e9b2545ff90e7ca30c80c0139',
   'RANK': '183896',
   'LOCALES': [],
   '__TYPE__': 'artist'},
  {'ART_ID': '14659541',
   'ROLE_ID': '0',
   'ARTISTS_SONGS_ORDER': '2',
   'ART_NAME': "Limsa d'Aulnay",
   'ARTIST_IS_DUMMY': False,
   'ART_PICTURE': 'fdf46f8f129ba2d3897e94cfa9bc80f9',
   'RANK': '659458',
   'LOCALES': {'lang_fr': {'name': "Limsa d'Aulnay"}},
   '__TYPE__': 'artist'}],
 'ALB_ID': '309761227',
 'ALB_TITLE': 'Gaz120',
 'TYPE': 0,
 'VIDEO': False,
 'DURATION': '168',
 'ALB_PICTURE': 'e4a3578e9b2545ff90e7ca30c80c0139',
 'ART_PICTURE': 'e4a3578e9b2545ff90e7ca30c80c0139',
 'RANK_SNG': '285062',
 

In [16]:
dict_music = {}

In [17]:
title_list = [] #Get all the title on what the artist performed
album_set = set() #Get all the album that the artist is on
collab_set = set() #Get all the possible collab -1
time_list = []
time_count = 0
collab = False

for title in part_to_parse :

    album_set.add(title["ALB_TITLE"])
    time_list.append(title["DURATION"])
    time_count += int(title["DURATION"])

    if len(title["ARTISTS"])>1 : 
        title_list.append([title["SNG_TITLE"], True])
        for artist in title["ARTISTS"] :
            collab_set.add(artist["ART_NAME"])
    else :
        title_list.append([title["SNG_TITLE"], False])

In [18]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import urllib

# --- Your search settings ---
artist = "Limsa d'Aulnay "
#title = title_list[0]
#query = artist + title
target_site = "genius.com"

In [19]:
def Accept_cookies_genius (driver) :

    # -- Cookies en 2 parties -- #
    try:
        wait = WebDriverWait(driver, 10)
        button_cookies = wait.until(EC.element_to_be_clickable((By.XPATH, '//button[text()="Afficher toutes les finalités"]')))
        button_cookies.click()
    except Exception as e :
        driver.quit()
        print("No button found")

    try:
        wait = WebDriverWait(driver, 10)
        button_cookies = wait.until(EC.element_to_be_clickable((By.XPATH, '//button[text()="Confirmer la sélection"]')))
        button_cookies.click()
    except Exception as e :
        driver.quit()
        print("No button found")

In [62]:
def Get_lyrics_genius (collab, driver) :

    src = driver.page_source
    new = bs(src,"html.parser")

    # On récupère tous les blocs qui contiennent les paroles
    lyrics_blocks = new.find_all("div", attrs={"data-lyrics-container": "true"})
    paroles = []
    author_lyrics = False

    if new.find_all("span", string = "Featuring") :
        collab = True
    else : 
        collab = False

    for block in lyrics_blocks:
        ligne = ""
        for elem in block.children:
            #print(elem)
            if collab and "[" in elem.text and ":" in elem.text:
                author_lyrics = "Limsa" in elem.text
            #    continue
            #if collab and "]" in elem.text :
            #    continue 
            
            if collab and not author_lyrics : #Passe au prochain elem jusqu'à ce qu'un élément corresponde à author_lyrics
                continue

            if elem.name == "br":
                    # Si <br> est seul (ligne vide), on saute
                if ligne.strip():
                    paroles.append(ligne.strip())
                ligne = ""
            elif elem.string:
                ligne += elem.string
            elif elem.name == "i" and "class" not in str(elem): #Doit exclure des class qui sont contenus dans des names <i>
                #Certains textes en italique ont alors l'indication <i> example : 
                #<i>Qui a encore un peu d’espoir ?<br/>J’préfère être mal accompagné plutôt qu’seul au sommet dans le brouillard</i>
                #Dans la section en italique, les sauts de ligne ne sont pas automatique dans le texte mais indiqué par <br/>
                tnf = re.sub(r"</?i\s*/?>", "", str(elem)) #retire <i> / </i> / </i/> / <i/>
                dnc = tnf.split("<br/>")
                for i in dnc : 
                    paroles.append(i.strip())
    # Ajouter la dernière ligne si elle existe
        if ligne.strip():
            paroles.append(ligne.strip())
    
    return paroles

In [ ]:
import time
import unicodedata

New

In [ ]:
len(set(title_list))

62

## Second part proper code

In [ ]:
r = page.find("h1", attrs={"id" : "naboo_artist_name"}).span.get_text(strip=True)
print(r)

Limsa d'Aulnay


In [ ]:
#Get the html page
request = requests.get(url="https://www.deezer.com/fr/artist/14659541/top_track")
html_page = request.text
page = bs(html_page,"html.parser")

#Get the artist name
artist_name = page.find("h1", attrs={"id" : "naboo_artist_name"}).span.get_text(strip=True)
print(artist_name)

#L'entiéreté de la discographie de l'artiste est contenue dans une gigantesque partie script 
res = page.find("script", string = re.compile("ART_ID"))

vvv = res.text.split("window.__DZR_APP_STATE__ = ")[1]
structured_text = json.loads(vvv)

In [74]:
title_list = [] #Get all the title on what the artist performed
album_set = set() #Get all the album that the artist is on
collab_set = set() #Get all the possible collab -1
time_list = []
time_count = 0
collab = False

for title in part_to_parse :

    album_set.add(title["ALB_TITLE"])
    time_list.append(title["DURATION"])
    time_count += int(title["DURATION"])

    if len(title["ARTISTS"])>1 : 
        title_list.append([title["SNG_TITLE"], True])
        for artist in title["ARTISTS"] :
            collab_set.add(artist["ART_NAME"])
    else :
        title_list.append([title["SNG_TITLE"], False])

In [75]:
def Accept_cookies_genius (driver) :

    # -- Cookies en 2 parties -- #
    try:
        wait = WebDriverWait(driver, 10)
        button_cookies = wait.until(EC.element_to_be_clickable((By.XPATH, '//button[text()="Afficher toutes les finalités"]')))
        button_cookies.click()
    except Exception as e :
        driver.quit()
        print("No button found")

    try:
        wait = WebDriverWait(driver, 10)
        button_cookies = wait.until(EC.element_to_be_clickable((By.XPATH, '//button[text()="Confirmer la sélection"]')))
        button_cookies.click()
    except Exception as e :
        driver.quit()
        print("No button found")


#Fonction GPT
def remove_accents(text):
    return ''.join(
        c for c in unicodedata.normalize('NFD', text)
        if unicodedata.category(c) != 'Mn'
    )


In [76]:
def Get_lyrics_genius (collab, driver) :

    src = driver.page_source
    new = bs(src,"html.parser")

    # On récupère tous les blocs qui contiennent les paroles
    lyrics_blocks = new.find_all("div", attrs={"data-lyrics-container": "true"})
    paroles = []
    author_lyrics = False

    if new.find_all("span", string = "Featuring") :
        collab = True
    else : 
        collab = False

    for block in lyrics_blocks:
        ligne = ""
        for elem in block.children:
            #print(elem)
            if collab and "[" in elem.text and ":" in elem.text:
                author_lyrics = "Limsa" in elem.text
            #    continue
            #if collab and "]" in elem.text :
            #    continue 
            
            if collab and not author_lyrics : #Passe au prochain elem jusqu'à ce qu'un élément corresponde à author_lyrics
                continue

            if elem.name == "br":
                    # Si <br> est seul (ligne vide), on saute
                if ligne.strip():
                    paroles.append(ligne.strip())
                ligne = ""
            elif elem.string:
                ligne += elem.string
            elif elem.name == "i" and "class" not in str(elem): #Doit exclure des class qui sont contenus dans des names <i>
                #Certains textes en italique ont alors l'indication <i> example : 
                #<i>Qui a encore un peu d’espoir ?<br/>J’préfère être mal accompagné plutôt qu’seul au sommet dans le brouillard</i>
                #Dans la section en italique, les sauts de ligne ne sont pas automatique dans le texte mais indiqué par <br/>
                tnf = re.sub(r"</?i\s*/?>", "", str(elem)) #retire <i> / </i> / </i/> / <i/>
                dnc = tnf.split("<br/>")
                for i in dnc : 
                    paroles.append(i.strip())
    # Ajouter la dernière ligne si elle existe
        if ligne.strip():
            paroles.append(ligne.strip())
    
    return paroles

In [113]:
def Navigate_Web (artist_name, title_list) :

    dict_parole = {}
    found_site = 0
    no_found = []

    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")  # Optional
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

    for idx, combo in tqdm(enumerate(title_list), total=len(title_list)):
        title = combo[0]
        collab = combo[1]
#        print(collab)
        query = artist_name + str(title)
        target_site = "genius.com"

#        print(f"Query for {title}")
        query = f"{query} site:{target_site}"
        url = "https://duckduckgo.com/?q=" + urllib.parse.quote(query)
        driver.get(url)
        
        WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'a'))
        )
        results = driver.find_elements(By.CSS_SELECTOR, 'a') #Store all the results

        target = "https://"+target_site
        found = False

        #Preparation du titre 
        title_adapt = re.sub(r"\([^)]*\)" , "" , (re.sub(r"[\'\.\-\°]", "", title))).lower()        
#        title_adapt = re.sub(r"\([^)]*\)", "", title.lower())
        pattern = re.sub(r"\s+", r"[-_+%20]*", (remove_accents(title_adapt)))

        for link in results :
            href = link.get_attribute("href")
            if href :
                last_word_link = href.split("-")[-1]
                #if target in href :
                    #print(href)
            if href and (target in href) and ((re.search(pattern,href.lower())) or (re.search(title.lower(),href.lower()))) and last_word_link == "lyrics" :
#                print(href)
                link.click()
                found = True
                found_site +=1
                break
        if not found : #Site Genius musique non trouvé
            no_found.append(f"No site found for {title}")
            #print(pattern)
            #print(url)
            found = False
            continue
    
        #print(f"Get to genius for {title}\n")
        if idx == 0 : 
            Accept_cookies_genius(driver)

        lyrics = Get_lyrics_genius(collab, driver)

        only_lyrics = []
        for i in lyrics: #A ce stade, les indications de couplet/refrains ont toujours là entre [] et les backs aussi entre ()
            i = re.sub(r"\([^)]*\)", "", i)       # retire les backs
            #i = re.sub(r"\[[^)]*\]", "", i)       # retire les indications 
            indication = ("couplet" in i.lower()) or ("refrain" in i.lower()) or ("intro" in i.lower()) or ("paroles" in i.lower()) 
            if i and indication == False: #Contains at least an element
                only_lyrics.append(i)    
        
        dict_parole[f"{title}"] = only_lyrics
    
    print("Finish")
    print(f"{found_site} on {len(title_list)} songs found")
    for i in no_found : 
        print(i)
    driver.quit()

    return dict_parole

In [114]:
artist = "Limsa d'Aulnay "
target_site = "genius.com"

In [115]:
result = Navigate_Web(artist, title_list)

100%|██████████| 62/62 [03:39<00:00,  3.54s/it]


Finish
54 on 62 songs found
No site found for 20M³
No site found for 93° FAHRENHEIT
No site found for Black Room - Bonus Track
No site found for Champions
No site found for Pas trop mal
No site found for Bendo
No site found for Ptg#2 (pourquoi tu guettes)
No site found for La hess


Après inspection, certains caractères non voulu reste comme \u2005 qui indiquerait un espace quatre-per-em mais aussi certain \ qui m'ont l'air d'être juste avant une apostrophe 

In [116]:
for title in list(result.keys()): 
    lines = result[title] #Je suis obligé d'itérer sur une copie et nan le vrai dict car je modifie sa valeur si len(lines) == 0
    
    lines = [line.replace("\u2005", " ") for line in lines]
    lines = [line.replace("[", "") for line in lines]
    lines = [line.replace("]", "") for line in lines]
    
    result[title] = lines

    if not lines:
        del result[title]
        print(f"Deleted: {title}")


Deleted: S.D.E
Deleted: P. Diddy


P. Diddy n'a pas de paroles car il s'agit d'un featuring et qu'aucune instruction n'est donnée sur qui chante quelle partie donc il n'est pas possible d'assigner les paroles avec certitude
De même pour S.D.E qui est indiqué en featuring mais en réalité DJ.Weedim ne chante pas donc aucune parole n'est assignée explicitement à Limsa D'Aulnay

In [117]:
for title in result :
    print(result[title])

["J’ai grandi loin d'la beach et des palmiers, c'était par choix mais c'était pas l’mien", 'À vingt-deux, j’suis à la niche, au panier, j’ressors si j’ai ni shit ni champagne', "J’suis en pleine créa', le projet prêt à leaker", 'Rap réalité comme BEN', 'Infamous Mobb, ils font les connaisseurs, en fait, ils nous snobent', 'Gros pét’ au bec d’vant les allocations', 'D’la valeur d’un crédit à la consommation', 'Vin sec, rosé, j’fais une insolation', 'Vingt-cinq degrés, j’fais une coloration', 'Rhum à 50 degrés pour ton information', 'On est v’nus tout baiser dans les prolongations', 'Qui a du feu ? Faut qu’j’en fasse un comme Mufasa', "Vaut mieux être seul au sommet qu'être accompagné de fantassins", 'Qui a des feuilles ? Car il m’en faut', 'Pour oublier mon frère à Fresnes, et tous ces trucs sentimentaux', 'Qui a encore un peu d’espoir ?', 'J’préfère être mal accompagné plutôt qu’seul au sommet dans le brouillard', 'J’avais la notice, j’ai pas lu, j’devais t’oublier, j’ai pas pu du tout

In [134]:
#Regroupement de l'entiéreté du corpus de texte
corpus = []

for title in result :
    for elem in result[title] :
        corpus.extend([elem])

In [137]:
#Préparation à la tokenization
corpus_RNN = "\n".join(corpus)
corpus_RNN = corpus_RNN.replace('"',"")
corpus_tokenization = corpus_RNN.replace(',',"")
corpus_tokenization = corpus_tokenization.replace('?',"")

In [142]:
pip install fasttext-wheel

     ---------------------------------------- 0.0/241.5 kB ? eta -:--:--
     -------------------------------------- 241.5/241.5 kB 7.5 MB/s eta 0:00:00
  Using cached pybind11-3.0.0-py3-none-any.whl (292 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [143]:
import fasttext

# Skipgram model :
model = fasttext.train_unsupervised('corpus_tokenization.txt', model='skipgram')

In [147]:
essao = ["c'est des rimes, le deuxième truc qu'ça m'inspire, c'est des blagues ", "Toute ma vie, j'écoute des packs de prods", "La nuit, j'écris des vieilles berceuses, c'est bien quand c'est bre-som, quand c'est ténébreux ", "On a tous une cible dans l'dos, la mort arrive comme un missile à tête chercheuse", "Des fois, j'déconne, j'abandonne vite, j'suis pas assez fort et j'crame des clopes", "En pensant avec quel objet j'vais exploser le crâne des opps", "On sait c'qu'on dit quand on parle du bitume, on sait c'qu'on dit quand on parle des halls", "Et toutes les meufs autour de nous, nan, c'est pas des putes qui kiffent la violence ", 'Les embrouilles, elles trouvent ça médiocre ', "Sauf pour défoncer deux-trois tarés qu'arborent fièrement leur croix gammée", "Quand j'vois un micro, je dois l'cramer, quand j'vois un mytho, je dois l'cramer", 'Babtou, rebeu, renoi, cramer', "J'ai appelé Manny deux fois  et il m'ramène de quoi m'calmer", 'Eh, eh', "Sauf si t'es prête à m'suivre quand l'Enfer m'appelle, ton amour, qu'est-c'que j'vais en faire, ma belle ?", "Je sors beaucoup, je m'enferme à peine, verre de R, grand bédo en plein après-m'", "J'viens d'là où ça rappe comme Pusha T ou Havoc, où ça tire, où ça vole, où ça deale", 'Où la seule devise, c\'est : "Sois riche ou sois digne"', "Outro : Limsa d'Aulnay", 'Où la seule devise, c\'est : "Sois riche ou sois digne"', 'Où la seule devise, c\'est : "Sois riche ou sois digne"', 'Où la seule devise, c\'est : "Sois riche ou sois digne"', 'Où la seule devise, c\'est : "Sois riche ou sois digne"', 'Où la seule devise, c\'est : "Sois riche ou sois digne"', 'Où la seule devise, c\'est : "Sois riche ou sois digne"', 'Où la seule devise, c\'est : "Sois riche ou sois digne"', 'Où la seule devise, c\'est : "Sois riche ou sois digne"', 'Où la seule devise, c\'est : "Sois riche ou sois digne"', 'Où la seule devise, c\'est : "Sois riche ou sois digne"', 'Où la seule devise, c\'est : "Sois riche ou sois digne"', 'Où la seule devise, c\'est : "Sois riche ou sois digne"', 'Où la seule devise, c\'est : "Sois riche ou sois digne"', 'Où la seule devise, c\'est : "Sois riche ou sois digne"']


In [151]:
corpus_tokenization

"J’ai grandi loin d'la beach et des palmiers c'était par choix mais c'était pas l’mien\nÀ vingt-deux j’suis à la niche au panier j’ressors si j’ai ni shit ni champagne\nJ’suis en pleine créa' le projet prêt à leaker\nRap réalité comme BEN\nInfamous Mobb ils font les connaisseurs en fait ils nous snobent\nGros pét’ au bec d’vant les allocations\nD’la valeur d’un crédit à la consommation\nVin sec rosé j’fais une insolation\nVingt-cinq degrés j’fais une coloration\nRhum à 50 degrés pour ton information\nOn est v’nus tout baiser dans les prolongations\nQui a du feu  Faut qu’j’en fasse un comme Mufasa\nVaut mieux être seul au sommet qu'être accompagné de fantassins\nQui a des feuilles  Car il m’en faut\nPour oublier mon frère à Fresnes et tous ces trucs sentimentaux\nQui a encore un peu d’espoir \nJ’préfère être mal accompagné plutôt qu’seul au sommet dans le brouillard\nJ’avais la notice j’ai pas lu j’devais t’oublier j’ai pas pu du tout\nY a tous ces trucs sentimentaux\nDe l’autre côté du

In [150]:
fasttext.tokenize(corpus_tokenization)

['J’ai',
 'grandi',
 'loin',
 "d'la",
 'beach',
 'et',
 'des',
 'palmiers',
 "c'était",
 'par',
 'choix',
 'mais',
 "c'était",
 'pas',
 'l’mien',
 '</s>',
 'À',
 'vingt-deux',
 'j’suis',
 'à',
 'la',
 'niche',
 'au',
 'panier',
 'j’ressors',
 'si',
 'j’ai',
 'ni',
 'shit',
 'ni',
 'champagne',
 '</s>',
 'J’suis',
 'en',
 'pleine',
 "créa'",
 'le',
 'projet',
 'prêt',
 'à',
 'leaker',
 '</s>',
 'Rap',
 'réalité',
 'comme',
 'BEN',
 '</s>',
 'Infamous',
 'Mobb',
 'ils',
 'font',
 'les',
 'connaisseurs',
 'en',
 'fait',
 'ils',
 'nous',
 'snobent',
 '</s>',
 'Gros',
 'pét’',
 'au',
 'bec',
 'd’vant',
 'les',
 'allocations',
 '</s>',
 'D’la',
 'valeur',
 'd’un',
 'crédit',
 'à',
 'la',
 'consommation',
 '</s>',
 'Vin',
 'sec',
 'rosé',
 'j’fais',
 'une',
 'insolation',
 '</s>',
 'Vingt-cinq',
 'degrés',
 'j’fais',
 'une',
 'coloration',
 '</s>',
 'Rhum',
 'à',
 '50',
 'degrés',
 'pour',
 'ton',
 'information',
 '</s>',
 'On',
 'est',
 'v’nus',
 'tout',
 'baiser',
 'dans',
 'les',
 'prolong

In [156]:
model.get_nearest_neighbors("voiture")

[(0.9998944401741028, "j'croyais"),
 (0.9998873472213745, "j'sais"),
 (0.9998850226402283, 'rentre'),
 (0.9998849630355835, 'sais'),
 (0.9998811483383179, 'chaque'),
 (0.9998809099197388, 'Dustin'),
 (0.9998804330825806, 'maille'),
 (0.9998804330825806, "j'les"),
 (0.9998802542686462, 'mais'),
 (0.9998794794082642, "p'tites")]

In [152]:
model.words

['</s>',
 'les',
 'des',
 'la',
 'pas',
 'de',
 'comme',
 'à',
 'un',
 'et',
 'le',
 'a',
 'on',
 'dans',
 'du',
 'que',
 'en',
 'je',
 "c'est",
 'qui',
 'pour',
 'ça',
 "j'suis",
 'plus',
 'mon',
 "j'ai",
 'fait',
 'une',
 'mais',
 'On',
 'ouais',
 'si',
 'quand',
 'au',
 'avec',
 'tu',
 'y',
 'par',
 'tout',
 'sur',
 'ou',
 'gros',
 'Et',
 'Ouais',
 'ma',
 'faire',
 'quelques',
 'Mais',
 ':',
 'est',
 'il',
 "J'ai",
 'mes',
 'moi',
 'me',
 'Les',
 "qu'on",
 'même',
 "J'suis",
 'eh',
 'À',
 'elle',
 'Quand',
 'dit',
 'Limsa',
 'Si',
 'pote',
 'woof',
 'bien',
 "p'tit",
 'fois',
 'ils',
 'Eh',
 'était',
 'footballeur',
 'aux',
 "t'es",
 'La',
 'Y',
 'deux',
 'Tu',
 'ASB',
 'être',
 'trop',
 'connu',
 'ont',
 "d'la",
 'où',
 "p'tits",
 'sont',
 'Gros',
 'Ça',
 'Le',
 'ai',
 "c'que",
 'font',
 'faut',
 'tous',
 "j'étais",
 'Je',
 "j'viens",
 'night',
 'mal',
 'son',
 'rien',
 'va',
 'vie',
 'sac',
 'rue',
 'sans',
 'Des',
 'gars',
 'fais',
 "j'en",
 "t'as",
 "j'vais",
 'tête',
 'ton',
 "

In [138]:
with open("corpus_RNN.txt", "w", encoding="utf-8") as f:
    f.write(corpus_RNN)

with open("corpus_tokenization.txt", "w", encoding="utf-8") as f:
    f.write(corpus_tokenization)

In [124]:
corpus

["J’ai grandi loin d'la beach et des palmiers, c'était par choix mais c'était pas l’mien",
 'À vingt-deux, j’suis à la niche, au panier, j’ressors si j’ai ni shit ni champagne',
 "J’suis en pleine créa', le projet prêt à leaker",
 'Rap réalité comme BEN',
 'Infamous Mobb, ils font les connaisseurs, en fait, ils nous snobent',
 'Gros pét’ au bec d’vant les allocations',
 'D’la valeur d’un crédit à la consommation',
 'Vin sec, rosé, j’fais une insolation',
 'Vingt-cinq degrés, j’fais une coloration',
 'Rhum à 50 degrés pour ton information',
 'On est v’nus tout baiser dans les prolongations',
 'Qui a du feu ? Faut qu’j’en fasse un comme Mufasa',
 "Vaut mieux être seul au sommet qu'être accompagné de fantassins",
 'Qui a des feuilles ? Car il m’en faut',
 'Pour oublier mon frère à Fresnes, et tous ces trucs sentimentaux',
 'Qui a encore un peu d’espoir ?',
 'J’préfère être mal accompagné plutôt qu’seul au sommet dans le brouillard',
 'J’avais la notice, j’ai pas lu, j’devais t’oublier, j’

In [49]:
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")  # Optional
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

query = f"Limsa d'Aulnay Trucs sentimentaux site:{target_site}"
url = "https://duckduckgo.com/?q=" + urllib.parse.quote(query)
driver.get(url)

In [50]:
src = driver.page_source
new = bs(src,"html.parser")

In [55]:
if new.find_all("span", string = "Featuring") :
    print("oui")

oui


In [ ]:
def Get_lyrics_genius (collab, driver) :

    src = driver.page_source
    new = bs(src,"html.parser")

    # On récupère tous les blocs qui contiennent les paroles
    lyrics_blocks = new.find_all("div", attrs={"data-lyrics-container": "true"})
    paroles = []
    author_lyrics = False
    

    for block in lyrics_blocks:
        ligne = ""
        for elem in block.children:
            #print(elem)
            if collab and "[" in elem.text and ":" in elem.text:
                author_lyrics = "Limsa" in elem.text
            #    continue
            #if collab and "]" in elem.text :
            #    continue 
            print(elem.text, author_lyrics)
            
            if collab and not author_lyrics : #Passe au prochain elem jusqu'à ce qu'un élément corresponde à author_lyrics
                continue

            if elem.name == "br":
                    # Si <br> est seul (ligne vide), on saute
                if ligne.strip():
                    paroles.append(ligne.strip())
                ligne = ""
            elif elem.string:
                ligne += elem.string
            elif elem.name == "i" and "class" not in str(elem): #Doit exclure des class qui sont contenus dans des names <i>
                #Certains textes en italique ont alors l'indication <i> example : 
                #<i>Qui a encore un peu d’espoir ?<br/>J’préfère être mal accompagné plutôt qu’seul au sommet dans le brouillard</i>
                #Dans la section en italique, les sauts de ligne ne sont pas automatique dans le texte mais indiqué par <br/>
                tnf = re.sub(r"</?i\s*/?>", "", str(elem)) #retire <i> / </i> / </i/> / <i/>
                dnc = tnf.split("<br/>")
                for i in dnc : 
                    paroles.append(i.strip())
    # Ajouter la dernière ligne si elle existe
        if ligne.strip():
            paroles.append(ligne.strip())
    
    return paroles

In [29]:
Get_lyrics_genius (True,driver)

3 ContributorsBougies Lyrics False
[Paroles de "Bougies" ft. Limsa d'Aulnay] False
 False
 False
[Refrain : Absolem] False
 False
Quelques bougies soufflées, quelques projets dans l'vent False
 False
Alors, dis-moi c'qu'il nous reste de nos rêves d'enfant False
 False
J'ai causé quelques blessures, collé quelques pansements False
 False
Certaines de tes pensées qu'j'aurais aimé entendre False
 False
Parfois, j'ai pas géré, j'suis pas mauvais dans l'fond False
 False
J't'ai causé quelques blessures, collé quelques pansements False
 False
 False
[Couplet 1 : Absolem] False
 False
Ça dit quoi depuis tout c'temps ? J'suis un peu moins insouciant False
 False
Toujours les mêmеs tours dans les mêmes coins, rien dе ressourssant False
 False
J'ai toujours les yeux rouges sang, j'm'endors en fumant, j'me réveille en toussant (uh) False
 False
J'ai perdu des bons amis, j'ai gardé quelques mauvais réflexes False
 False
Putain, comment on aurait flex si y avait les billets mauves et les racks Fals

["[Couplet 2 : Limsa d'Aulnay]",
 "Pour le cash et les putes, j'y vais, j'bois du [Boum Boum ?], pas du JB",
 'Sur ma ble-ta, y a du humus, y a du gibier (ouf)',
 'On a connu les rues bres-som, se plaindre, seules les putes le font (bah ouais)',
 "Toi, t'appelles ça une galère, c'est juste une épreuve, juste une leçon",
 "J'suis mauvais en négociation, mon style de vie est trop provocant",
 "Quand j'fais la somme de mes actions, l'Enfer, j'vais y aller en toboggan",
 "Le changement est trop choquant : avant, j'adorais bavarder",
 "J'traînais comme un attardé, maintenant, j'dis que j'vais pas tarder",
 "Mes rêves de star vont jamais s'accomplir donc j'dois pas laisser mes échecs m'assombrir",
 'En plus, chaque chose a son prix (Logique)',
 "On s'demandait c'qu'on allait faire de moi, maintenant, tout l'monde est fier de moi",
 "Et ça m'déprime parce que j'vois pas c'que j'vais faire de mieux, c'que j'vais faire demain"]

In [20]:
query = f"Limsa d'Aulnay Trucs sentimentaux site:{target_site}"
url = "https://duckduckgo.com/?q=" + urllib.parse.quote(query)
driver.get(url)

In [21]:
results = driver.find_elements(By.CSS_SELECTOR, 'a')

In [22]:
target = "https://"+target_site
for link in results :
    href = link.get_attribute("href")
    if href and target in href :
        print(href)
        link.click()
        break

https://genius.com/Ben-plg-trucs-sentimentaux-lyrics


In [23]:
try:
    wait = WebDriverWait(driver, 10)
    button_cookies = wait.until(EC.element_to_be_clickable((By.XPATH, '//button[text()="Afficher toutes les finalités"]')))
    button_cookies.click()
except Exception as e :
    driver.quit()
    print("No button found")

In [24]:
try:
    wait = WebDriverWait(driver, 10)
    button_cookies = wait.until(EC.element_to_be_clickable((By.XPATH, '//button[text()="Confirmer la sélection"]')))
    button_cookies.click()
except Exception as e :
    driver.quit()
    print("No button found")

In [25]:
src = driver.page_source
new = bs(src,"html.parser")

In [1]:
def Get_lyrics_genius (collab, driver) :

    src = driver.page_source
    new = bs(src,"html.parser")

    # On récupère tous les blocs qui contiennent les paroles
    lyrics_blocks = new.find_all("div", attrs={"data-lyrics-container": "true"})
    paroles = []
    author_lyrics = not collab

    for block in lyrics_blocks:
        ligne = ""
        for elem in block.children:
            #print(elem)
            if collab and "[" in elem.text and ":" in elem.text :
                author_lyrics = "Limsa" in elem.text
            #    continue
            #if collab and "]" in elem.text :
            #    continue 
            
            if not author_lyrics : #Passe au prochain elem jusqu'à ce qu'un élément corresponde à author_lyrics
                continue

            if elem.name == "br":
                    # Si <br> est seul (ligne vide), on saute
                if ligne.strip():
                    paroles.append(ligne.strip())
                ligne = ""
            elif elem.string:
                ligne += elem.string
            elif elem.name == "i" and "class" not in str(elem): #Doit exclure des class qui sont contenus dans des names <i>
                #Certains textes en italique ont alors l'indication <i> example : 
                #<i>Qui a encore un peu d’espoir ?<br/>J’préfère être mal accompagné plutôt qu’seul au sommet dans le brouillard</i>
                #Dans la section en italique, les sauts de ligne ne sont pas automatique dans le texte mais indiqué par <br/>
                tnf = re.sub(r"</?i\s*/?>", "", str(elem)) #retire <i> / </i> / </i/> / <i/>
                dnc = tnf.split("<br/>")
                for i in dnc : 
                    paroles.append(i.strip())
    # Ajouter la dernière ligne si elle existe
        if ligne.strip():
            paroles.append(ligne.strip())
    
    return paroles

In [2]:
paroles = Get_lyrics_genius(True,driver)
paroles

NameError: name 'driver' is not defined

In [114]:
only_lyrics = []
for i in paroles:
    i = re.sub(r"\([^)]*\)", "", i)       # remove (parentheses)
    i = re.sub(r"\[[^)]*\]", "", i)       # remove [brackets]
    if i : #Contains at least an element
        only_lyrics.append(i)    
only_lyrics

["J’ai grandi loin d'la beach et des palmiers, c'était par choix mais c'était pas l’mien",
 'À vingt-deux, j’suis à la niche, au panier, j’ressors si j’ai ni shit ni champagne',
 "J’suis en pleine créa', le projet prêt à leaker",
 'Rap réalité comme BEN',
 'Infamous Mobb, ils font les connaisseurs, en fait, ils nous snobent',
 'Gros pét’ au bec d’vant les allocations',
 'D’la valeur d’un crédit à la consommation',
 'Vin sec, rosé, j’fais une insolation',
 'Vingt-cinq degrés, j’fais une coloration',
 'Rhum à 50 degrés pour ton information',
 'On est v’nus tout baiser dans les prolongations',
 'Qui a du feu ? Faut qu’j’en fasse un comme Mufasa',
 "Vaut mieux être seul au sommet qu'être accompagné de fantassins",
 'Qui a des feuilles ? Car il m’en faut',
 'Pour oublier mon frère à Fresnes, et tous ces trucs sentimentaux',
 'Qui a encore un peu d’espoir ?',
 'J’préfère être mal accompagné plutôt qu’seul au sommet dans le brouillard',
 'J’avais la notice, j’ai pas lu, j’devais t’oublier, j’

In [115]:
dict_chanson = {"Trucs sentimentaux" : only_lyrics}

In [116]:
dict_chanson

{'Trucs sentimentaux': ["J’ai grandi loin d'la beach et des palmiers, c'était par choix mais c'était pas l’mien",
  'À vingt-deux, j’suis à la niche, au panier, j’ressors si j’ai ni shit ni champagne',
  "J’suis en pleine créa', le projet prêt à leaker",
  'Rap réalité comme BEN',
  'Infamous Mobb, ils font les connaisseurs, en fait, ils nous snobent',
  'Gros pét’ au bec d’vant les allocations',
  'D’la valeur d’un crédit à la consommation',
  'Vin sec, rosé, j’fais une insolation',
  'Vingt-cinq degrés, j’fais une coloration',
  'Rhum à 50 degrés pour ton information',
  'On est v’nus tout baiser dans les prolongations',
  'Qui a du feu ? Faut qu’j’en fasse un comme Mufasa',
  "Vaut mieux être seul au sommet qu'être accompagné de fantassins",
  'Qui a des feuilles ? Car il m’en faut',
  'Pour oublier mon frère à Fresnes, et tous ces trucs sentimentaux',
  'Qui a encore un peu d’espoir ?',
  'J’préfère être mal accompagné plutôt qu’seul au sommet dans le brouillard',
  'J’avais la not

In [120]:
dict_chanson["NN"] = only_lyrics

In [ ]:
Dict_chanson = {}